<a href="https://colab.research.google.com/github/Haniiko1/img_analysis/blob/main/image_analysis_fluorescent_with_interactive_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Cell 1: Install Libraries
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, filters, morphology, segmentation, measure, color
from skimage.util import img_as_ubyte
from scipy import ndimage
import pandas as pd
from ipywidgets import IntSlider, Output, VBox, HBox, Button, Dropdown, FloatSlider # Added Dropdown and FloatSlider
from IPython.display import display, clear_output, HTML
from ipywidgets import Layout # Import Layout for custom sizing
from skimage.feature import peak_local_max
from skimage.color import label2rgb # Import label2rgb for visualization
import json
import base64
import io
from PIL import Image
from skimage.measure import find_contours  # Import continuous contour processing pathfinder
from google.colab import files, output
from skimage import io as sk_io # Use alias to avoid conflict with standard io
import IPython

# Converting to a Grayscale Image
This helps eliminate noises and aggregates' intensity calculation. This cell below will prompt you to upload your microscopic image and convert it to a grayscale image automatically.

In [ ]:
#@title Cell 2: Convert to a grayscale image

print("Please upload your image file:")
uploaded = files.upload()

# Assuming only one file is uploaded, get its name and content
if uploaded:
    for filename, content in uploaded.items():
        image_data = io.BytesIO(content)
        temp_img_loaded = sk_io.imread(image_data)
        print(f"Uploaded file: {filename}")
        break # Process only the first uploaded file
else:
    raise ValueError("No image file was uploaded.")

original_rgb_img = temp_img_loaded.copy() # Store the original RGB image

print(f"shape: {original_rgb_img.shape}, dtype: {original_rgb_img.dtype}")

# If RGB, extract green channel for downstream processing (img and img_16bit)
if temp_img_loaded.ndim == 3:
    img = temp_img_loaded[:, :, 1]  # green channel
else:
    img = temp_img_loaded # If already grayscale, just assign

# Show the extracted green channel here as a standard fluorescence image
plt.figure(figsize=(6,6))
plt.imshow(img, cmap='gray')
plt.title('Original Image (green channel extracted)')
plt.axis('off')
plt.show()
print(f"min: {img.min()}, max: {img.max()}")

# Convert to a 16-bit Image
 The saved JPG is a compressed version of the microscopic image. The conversion into a 16-bit format allows more level of gray (65,536 levels of gray in 16-bit vs. 256 in 8-bit).

In [ ]:
#@title Cell 3: Convert to a 16-bit image
img_16bit = img.astype(np.uint16)

plt.figure(figsize=(6,6))
plt.imshow(img_16bit, cmap='gray')
plt.title('16-bit Image')
plt.axis('off')
plt.colorbar(label='Intensity (0–65535)')
plt.show()
print(f"dtype: {img_16bit.dtype}, min: {img_16bit.min()}, max: {img_16bit.max()}")

In [ ]:
#@title Find background intensity
if 'thresholded' not in globals() or thresholded.shape != img_16bit.shape:
    print("Warning: 'thresholded' not found or its shape is inconsistent with 'img_16bit'. Re-calculating using Yen's method.")
    thresh_val = filters.threshold_yen(img_16bit)
    thresholded = img_16bit > thresh_val
else:
    print("Using 'thresholded' from previous definition (e.g., Cell 4: Set a threshold).")

background_mask = ~thresholded # Invert the mask to get background pixels
background_intensities = img_16bit[background_mask]

mean_background_intensity = np.mean(background_intensities)
std_background_intensity = np.std(background_intensities)

print(f"Mean background intensity: {mean_background_intensity:.2f}")
print(f"Standard deviation of background intensity: {std_background_intensity:.2f}")

# Set a Threshold
 This is the core segmentation step — you're drawing a line between "signal" (aggregates) and "background" (everything else).

In [ ]:
#@title Cell 4: Set a threshold


# Get the min and max intensity values from the image for slider range
img_min = img_16bit.min()
img_max = img_16bit.max()

# Dictionary of thresholding methods
threshold_methods = {
    'Otsu': filters.threshold_otsu,
    'Yen': filters.threshold_yen,
    'Li': filters.threshold_li,
    'Isodata': filters.threshold_isodata,
    'Triangle': filters.threshold_triangle,
}

# Dropdown for thresholding method selection
method_dropdown = Dropdown(
    options=list(threshold_methods.keys()),
    value='Yen',
    description='Method:',
    layout=Layout(width='200px')
)

# Function to calculate threshold based on selected method
def calculate_threshold(method_name):
    method_func = threshold_methods[method_name]
    try:
        return method_func(img_16bit)
    except RuntimeError as e:
        print(f"Warning: {method_name} method failed for this image. {e}. Defaulting to Otsu's.")
        return filters.threshold_otsu(img_16bit)

# Calculate initial threshold using the default method (Yen)
initial_method = method_dropdown.value
current_thresh_val = calculate_threshold(initial_method)
print(f"{initial_method}'s suggested threshold: {current_thresh_val:.0f} (Image min: {img_min}, max: {img_max})")

# --- Create the plot elements once ---
fig, axes = plt.subplots(1, 2, figsize=(16.8, 7))

# Plot the 'Before Threshold' image (this part is static)
axes[0].imshow(img_16bit, cmap='gray')
axes[0].set_title('Before Threshold')
axes[0].axis('off')

# Initialize the 'After Threshold' image (this will be updated)
# Use initial calculated threshold for the first display
initial_thresholded = img_16bit > int(current_thresh_val)
normalized_img_initial = img_16bit.astype(float) / img_max
overlay_initial = color.gray2rgb(img_as_ubyte(normalized_img_initial))
overlay_initial[initial_thresholded] = [255, 0, 0]
im_after = axes[1].imshow(overlay_initial) # Store reference to the imshow object
axes[1].set_title(f'After Threshold (red = selected, val={int(current_thresh_val):.0f})')
axes[1].axis('off')

plt.tight_layout()

# Output widget to display the figure (for in-place updates)
plot_output = Output()
with plot_output:
    display(fig) # Display the figure within the output widget
    plt.close(fig) # Close the figure to prevent duplicate display by the notebook

# Output widget to display text feedback (e.g., current threshold value)
text_output = Output()

# Global variables to store current threshold and thresholded mask
global thresh_val, thresholded

def update_threshold_plot_artists(change=None, manual_val=None):
    global thresh_val, thresholded

    if manual_val is not None:
        # Manual slider update
        thresh_val = manual_val
        current_method_name = "Manual"
    else:
        # Method dropdown update or initial call
        current_method_name = method_dropdown.value
        thresh_val = calculate_threshold(current_method_name)

    thresholded = img_16bit > thresh_val # Update global thresholded mask

    with text_output:
        clear_output(wait=True) # Clear previous text output
        print(f"Current Threshold Method: {current_method_name}")
        print(f"Current Threshold Value: {thresh_val:.0f}")

    # Calculate the new overlay image
    normalized_img_current = img_16bit.astype(float) / img_max
    overlay_current = color.gray2rgb(img_as_ubyte(normalized_img_current))
    overlay_current[thresholded] = [255, 0, 0] # Use the global `thresholded`

    # Update the data of the existing imshow object
    im_after.set_data(overlay_current)
    axes[1].set_title(f'After Threshold (red = selected, val={thresh_val:.0f})')

    # Force redraw by clearing and redisplaying the figure within the output widget
    with plot_output:
        clear_output(wait=True)
        display(fig)

# Create an interactive slider for manual threshold adjustment
threshold_slider = IntSlider(
    value=int(current_thresh_val), # Initialize with the calculated threshold
    min=int(img_min),
    max=int(img_max),
    step=1,
    description='Threshold:',
    continuous_update=True,
    layout=Layout(width='500px') # Make the slider bigger
)

# Create buttons for increment/decrement
increase_button = Button(description='+', layout=Layout(width='50px'))
decrease_button = Button(description='-', layout=Layout(width='50px'))

# Define button click handlers
def on_increase_button_clicked(b):
    threshold_slider.value = min(threshold_slider.max, threshold_slider.value + 1)
    # Call update_threshold_plot_artists with manual_val to indicate slider change
    update_threshold_plot_artists(manual_val=threshold_slider.value)

def on_decrease_button_clicked(b):
    threshold_slider.value = max(threshold_slider.min, threshold_slider.value - 1)
    # Call update_threshold_plot_artists with manual_val to indicate slider change
    update_threshold_plot_artists(manual_val=threshold_slider.value)

# Observe button clicks
increase_button.on_click(on_increase_button_clicked)
decrease_button.on_click(on_decrease_button_clicked)

# Observe changes in the slider value and call the update function
def on_slider_value_change(change):
    update_threshold_plot_artists(manual_val=change.new)

threshold_slider.observe(on_slider_value_change, names='value')

# Observe changes in the method dropdown and call the update function
def on_method_dropdown_change(change):
    new_threshold_value = calculate_threshold(change.new)
    threshold_slider.value = int(new_threshold_value) # Update slider to new method's value
    update_threshold_plot_artists(change=change) # Update plot based on new method

method_dropdown.observe(on_method_dropdown_change, names='value')

# Arrange slider and buttons horizontally, then combine with method dropdown and text output vertically
slider_and_buttons = HBox([decrease_button, threshold_slider, increase_button])
controls = VBox([method_dropdown, slider_and_buttons, text_output])

# Display the controls and the plot
display(controls, plot_output) # Display controls (slider + text) and the plot output

# Initialize thresh_val and thresholded globally based on the initial method.
# They will be updated by the slider or dropdown callbacks.
thresh_val = current_thresh_val
thresholded = img_16bit > thresh_val

# Initial plot update after all widgets are set up
update_threshold_plot_artists()

# Convert to a Binary Image
 Converting to binary reduces the image to pure black and white — pixels are either object (white) or background (black). This simplifies the image so downstream particle-counting algorithms can work cleanly, without being confused by intermediate gray values.

In [ ]:
#@title Cell 5 Convert to a binary image/black and white
binary = thresholded.copy()

plt.figure(figsize=(6,6))
plt.imshow(binary, cmap='binary_r')
plt.title('Binary Image')
plt.axis('off')
plt.show()

# Watershed
Most biologically important preprocessing step. In fluorescent aggregate images, aggregates that are physically close together often appear as a single fused white blob after thresholding. Watershed works by:


*   Finding intensity "peaks" within a blob (likely individual aggregate centers)
*   Drawing artificial boundaries between them




In [ ]:
#@title Cell 6: Watershed image


distance = ndimage.distance_transform_edt(binary)

# Use min_distance=1 to catch every individual peak
coords = peak_local_max(distance, min_distance=4, labels=binary)
local_max_mask = np.zeros(distance.shape, dtype=bool)
local_max_mask[tuple(coords.T)] = True

markers = measure.label(local_max_mask)
watershed_result = segmentation.watershed(-distance, markers, mask=binary)

plt.figure(figsize=(6,6))
# Use label2rgb to visualize distinct segmented regions
plt.imshow(label2rgb(watershed_result, bg_label=0)) # bg_label=0 ensures background remains black
plt.title('After Watershed (individual aggregates colored)')
plt.axis('off')
plt.show()

# IGNORE CELL 6.5

In [ ]:
#@title (IGNORE FOR NOW) Cell 6.5 Comparing Watershed min values
from skimage.feature import peak_local_max
from skimage.color import label2rgb

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
test_values = [1, 2, 3, 4, 5, 6]

for ax, md in zip(axes.flat, test_values):
    coords = peak_local_max(distance, min_distance=md, labels=binary)
    lm = np.zeros(distance.shape, dtype=bool)
    lm[tuple(coords.T)] = True
    markers = measure.label(lm)
    ws = segmentation.watershed(-distance, markers, mask=binary)

    # count after size filter
    props_test = [p for p in measure.regionprops(ws, intensity_image=img_16bit)
                  if 10 <= p.area <= 500]

    ax.imshow(label2rgb(ws, bg_label=0))
    ax.set_title(f'min_distance={md}  →  {len(props_test)} aggregates')
    ax.axis('off')

plt.tight_layout()
plt.show()

# Registering All of The Aggregates

In [ ]:


#@title Outlining the aggregates (takes a couple seconds to highlight on a high res)
# Mask out bottom-right corner where scale bar is
img_h, img_w = img_16bit.shape
mask = watershed_result.copy()
# Removed: mask[int(img_h * 0.85):, int(img_w * 0.75):] = 0

props = measure.regionprops(mask, intensity_image=img_16bit)

# --- OPTIMIZATION: Pre-calculate all contours once ---
# This dictionary will store contours for each region label, calculated once.
all_contours_precomputed = {}
for p in props:
    region_mask = (watershed_result == p.label)
    all_contours_precomputed[p.label] = measure.find_contours(region_mask, 0.5)
# --- END OPTIMIZATION ---

# Create the dropdown widget globally so it can be updated
all_labels = ['None'] + [str(p.label) for p in props] # Get all possible labels initially as strings
label_dropdown = Dropdown(
    options=all_labels,
    value='None', # Default to no highlighting
    description='Highlight Label:',
)

def update_plot(min_size, max_size, min_circularity, max_circularity, highlight_label=None):
    with plot_output:
        clear_output(wait=True);

        props_filtered = []
        filtered_labels = ['None'] # Option to not highlight anything
        for p in props:
            if p.perimeter == 0: # Avoid division by zero for objects with no perimeter
                circularity = 0
            else:
                circularity = (4 * np.pi * p.area) / (p.perimeter ** 2)

            if min_size <= p.area <= max_size and min_circularity <= circularity <= max_circularity:
                props_filtered.append(p)
                filtered_labels.append(str(p.label)) # Add label as string for dropdown

        # Temporarily unobserve to prevent re-triggering update_plot
        label_dropdown.unobserve(on_label_dropdown_change, names='value')

        current_selected_label = label_dropdown.value # Store current value
        label_dropdown.options = filtered_labels # Update options
        if current_selected_label in filtered_labels:
            label_dropdown.value = current_selected_label # Restore if still valid
        else:
            label_dropdown.value = 'None' # Reset to 'None' if previously selected is no longer valid

        # Re-observe the dropdown
        label_dropdown.observe(on_label_dropdown_change, names='value')

        fig, ax = plt.subplots(1, 1, figsize=(8, 8)) # Create a single subplot

        # Plot directly on the original image
        ax.imshow(original_rgb_img) # Use original_rgb_img (full RGB) without cmap or alpha

        for prop in props_filtered:
            # --- OPTIMIZATION: Retrieve pre-calculated contours ---
            contours = all_contours_precomputed.get(prop.label, []) # Get contours from pre-computed dict
            # --- END OPTIMIZATION ---

            contour_color = 'cyan'
            contour_linewidth = 1
            if highlight_label is not None and str(prop.label) == highlight_label:
                contour_color = 'red'
                contour_linewidth = 2 # Make highlighted contour thicker

            for contour in contours:
                ax.plot(contour[:, 1], contour[:, 0], linewidth=contour_linewidth, color=contour_color)

        ax.set_title(f'Detected Aggregates: {len(props_filtered)}')
        ax.axis('off')

        plt.tight_layout()
        plt.show()

# Create interactive sliders
min_size_slider = IntSlider(
    value=1, # Initial value based on previous change
    min=1,
    max=100, # Max area changed to 100 as requested by the user
    step=1,
    description='Min Size:',
    continuous_update=True
)

max_size_slider = IntSlider(
    value=2000, # Initial value, increased from 500
    min=10,
    max=2000, # Max area changed to 2000 as requested by the user
    step=1,
    description='Max Size:',
    continuous_update=True
)

min_circularity_slider = FloatSlider(
    value=0.0, # Initial value
    min=0.0,
    max=1.5, # Allow for values slightly over 1 due to pixelation
    step=0.01,
    description='Min Circularity:',
    continuous_update=True
)

max_circularity_slider = FloatSlider(
    value=5.0, # Initial value (increased from 1.5)
    min=0.1,
    max=5.0, # Upper bound for circularity values (increased from 2.0)
    step=0.01,
    description='Max Circularity:',
    continuous_update=True
)

# Output widget to display the figure
plot_output = Output()

# Link sliders to the update function
def on_slider_change(change):
    update_plot(
        min_size_slider.value,
        max_size_slider.value,
        min_circularity_slider.value,
        max_circularity_slider.value,
        label_dropdown.value # Pass the current dropdown value
    )

# New function for dropdown change
def on_label_dropdown_change(change):
    update_plot(
        min_size_slider.value,
        max_size_slider.value,
        min_circularity_slider.value,
        max_circularity_slider.value,
        change.new # New value of the dropdown
    )

min_size_slider.observe(on_slider_change, names='value')
max_size_slider.observe(on_slider_change, names='value')
min_circularity_slider.observe(on_slider_change, names='value')
max_circularity_slider.observe(on_slider_change, names='value')
label_dropdown.observe(on_label_dropdown_change, names='value') # Observe the dropdown

# Display the sliders and dropdown, then the initial plot
display(VBox([min_size_slider, max_size_slider, min_circularity_slider, max_circularity_slider, label_dropdown]), plot_output)

# Trigger initial plot display
update_plot(min_size_slider.value, max_size_slider.value, min_circularity_slider.value, max_circularity_slider.value, label_dropdown.value)

# 🎛️ Interactive Tracker HUD User Guide

Welcome to the Automated Otsu Thresholding & Aggregation Dashboard. This interactive workspace bridges your Python data-processing pipeline with a real-time web interface, allowing you to visually audit, verify, and manually override automated segmentation results.

---

### 🕹️ Interface Controls & Navigation

| Action | Control | Result |
| :--- | :--- | :--- |
| **Inspect Aggregate** | **Click** any card in the sidebar lanes | Centers the canvas crosshairs on that item's centroid and matches its profile details. |
| **Relocate Crosshairs** | **Click & Drag** the intersection of the blue target crosshairs | Smoothly shifts the target area, updating the high-magnification view window. |
| **Move Zoom Window** | **Click & Drag** anywhere inside the 4x Zoom Viewport box | Relocates the overlay window anywhere on the main canvas matrix to reveal hidden or covered structures. |
| **Manual Override** | **Drag & Drop** any item card between the three categorical lanes | Changes the item's classification status and updates data pools in real time. |

---

### 📊 Understanding the Categorical Lanes

The dashboard operates on an automated calibration engine, dividing items into three distinct pipelines based on dynamic **Otsu Cutoffs** and geometric criteria:

1. **`accepted` (Green)**
   * **Criteria:** Meets or exceeds strict automated threshold boundaries for both pixel surface area and mean signal intensity, while staying within acceptable circularity ranges.
   * **Overlay:** Rendered with a solid green boundary contour (`rgba(46, 204, 113, 0.8)`).
2. **`manual check` (Yellow)**
   * **Criteria:** Passes absolute minimum cutoffs but falls within a **15% borderline safety buffer** of the Otsu limits. These items are flagged for human verification to prevent false positives.
   * **Overlay:** Rendered with a solid orange boundary contour (`rgba(243, 156, 18, 0.8)`).
3. **`rejected` (Red)**
   * **Criteria:** Fails to meet the image-calibrated minimum area or intensity floor, or falls outside the rigid 0.15 to 5.00 circularity boundaries (often indicating microscopic noise, background artifacts, or edge clipping).
   * **Overlay:** Rendered with a translucent red boundary contour (`rgba(231, 76, 60, 0.5)`).

---

### 🧪 Dynamic Scoring Mechanics

For accepted aggregates, a composite ranking score ($S$) is computed using a weighted balance of three normalized structural vectors:

$$S = 0.40 \cdot S_{\text{area}} + 0.35 \cdot S_{\text{intensity}} + 0.25 \cdot S_{\text{circularity}}$$

* **Area Score ($S_{\text{area}}$):** Linear scaling relative to the maximum observed cluster size within the target sample pool.
* **Intensity Score ($S_{\text{intensity}}$):** Normalized distance proximity optimized toward the center mass of the dynamic range window:
$$\frac{\text{Otsu Min} + \text{Max Observed}}{2}$$
* **Circularity Score ($S_{\text{circularity}}$):** Normalized distribution optimizing toward the center of your established static shape parameters.

---

### 🔄 Live Notebook Kernel Synchronization

* **Real-Time Pipeline Updates:** Moving an item card into or out of the `accepted` lane instantly tells the backend Python kernel to rewrite its indexing structure.
* **Downstream Analysis:** The global variable **`live_accepted_df`** always stays perfectly synchronized with your dashboard layout. You can call it in any subsequent notebook cell to run downstream statistics, export CSVs, or plot curated metrics.

In [ ]:
#@title Draggable Tracker HUD with Interactive Zoom Viewport Relocation (99% Cumulative Intensity)


# --- 1. SET UP THE DATA PIPELINE & EXTRACT ALL RAW PROPS ---
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Fallback block if running isolated without live watershed_result variables
if 'watershed_result' not in globals() or 'props' not in globals():
    class DummyProp:
        def __init__(self, label, area, intensity, centroid, bbox):
            self.label = label
            self.area = area
            self.mean_intensity = intensity
            self.centroid = centroid
            self.bbox = bbox
            self.image = np.array([[0,1,1,0],[1,1,1,1],[1,1,1,1],[0,1,1,0]])
    props = [
        DummyProp(1, 150, 45, (100, 150), (95, 145, 105, 155)),
        DummyProp(2, 12, 15, (200, 300), (195, 295, 205, 305)),
        DummyProp(3, 800, 180, (400, 250), (380, 230, 420, 270)),
        DummyProp(4, 45, 28, (150, 180), (145, 175, 155, 185)),
        DummyProp(5, 4, 10, (50, 50), (48, 48, 52, 52))
    ]
    watershed_result = np.zeros((600, 800), dtype=np.int32)

raw_data = []
calculated_circularities = {}
precomputed_contours_json = {}

for p in props:
    if 'watershed_result' in globals() and watershed_result is not None:
        region_mask = (watershed_result == p.label)
        contours = find_contours(region_mask, 0.5)
    else:
        padded_image = np.pad(p.image, pad_width=1, mode='constant', constant_values=0)
        contours = find_contours(padded_image, level=0.5)

    if len(contours) == 0:
        vector_perimeter = 0.0
        raw_contour_points = []
    else:
        contour = max(contours, key=len)
        dx = np.diff(contour[:, 1])
        dy = np.diff(contour[:, 0])
        vector_perimeter = np.sum(np.sqrt(dx**2 + dy**2))

        if 'watershed_result' in globals() and watershed_result is not None:
            raw_contour_points = [[round(float(pt[1]), 1), round(float(pt[0]), 1)] for pt in contour]
        else:
            min_y, min_x, _, _ = p.bbox
            raw_contour_points = [[round(float(pt[1] + min_x - 1), 1), round(float(pt[0] + min_y - 1), 1)] for pt in contour]

    circularity = (4 * np.pi * p.area) / (vector_perimeter ** 2) if vector_perimeter > 0 else 0.0

    calculated_circularities[p.label] = round(circularity, 3)
    precomputed_contours_json[int(p.label)] = raw_contour_points

    raw_data.append({
        'label':          int(p.label),
        'area_px':        p.area,
        'mean_intensity': round(p.mean_intensity, 1),
        'centroid_y':     round(p.centroid[0], 1),
        'centroid_x':     round(p.centroid[1], 1),
        'circularity':    round(circularity, 3)
    })

df = pd.DataFrame(raw_data)

# --- 2. CALIBRATION ENGINE (OTSU AREA & 99% CUMULATIVE SUM INTENSITY) ---
def calculate_otsu_cutoff(values, use_log=False):
    if len(values) < 2:
        return np.mean(values) if len(values) == 1 else 0.0

    working_vals = np.log10(values + 1e-5) if use_log else np.array(values, dtype=np.float64)

    counts, bins = np.histogram(working_vals, bins='auto')
    bin_centers = (bins[:-1] + bins[1:]) / 2

    weight1 = np.cumsum(counts)
    weight2 = np.cumsum(counts[::-1])[::-1]

    with np.errstate(divide='ignore', invalid='ignore'):
        mean1 = np.cumsum(counts * bin_centers) / weight1
        mean2 = (np.cumsum((counts * bin_centers)[::-1]) / weight2[::-1])[::-1]
        variance_between = weight1[:-1] * weight2[1:] * (mean1[:-1] - mean2[1:]) ** 2

    valid_idx = np.isfinite(variance_between)
    if not np.any(valid_idx):
        return np.mean(values)

    best_idx = np.argmax(np.where(valid_idx, variance_between, -np.inf))
    cutoff = bin_centers[best_idx]

    return 10 ** cutoff if use_log else cutoff

# Compute image-specific dynamic boundary for Area using Otsu
OTSU_MIN_AREA = calculate_otsu_cutoff(df['area_px'].values, use_log=True)

# NEW: 99% Cumulative Sum Rule for Intensity
if not df.empty:
    # Sort intensities from highest to lowest
    sorted_intensities = np.sort(df['mean_intensity'].values)[::-1]
    total_intensity_mass = np.sum(sorted_intensities)

    if total_intensity_mass > 0:
        cumulative_sum = np.cumsum(sorted_intensities)
        fraction_sum = cumulative_sum / total_intensity_mass

        # Find the first index where we meet or exceed 99% of total signal intensity
        cutoff_idx = np.searchsorted(fraction_sum, 0.99)
        # Handle index clamping safety
        if cutoff_idx >= len(sorted_intensities):
            cutoff_idx = len(sorted_intensities) - 1

        # The cutoff value is the intensity at that specific cumulative point
        CUMULATIVE_MIN_INTENSITY = sorted_intensities[cutoff_idx]
    else:
        CUMULATIVE_MIN_INTENSITY = 0.0
else:
    CUMULATIVE_MIN_INTENSITY = 0.0

# Static Boundaries for Circularity
MIN_CIRCULARITY = 0.15
MAX_CIRCULARITY = 5.00

# Guard rails to clip scoring systems safely within matrix ranges
MAX_AREA = df['area_px'].max() if not df.empty else 2000
MAX_INTENSITY = df['mean_intensity'].max() if not df.empty else 255

print(f"--- AUTOMATED IMAGE CALIBRATION ---")
print(f"Dynamic Min Area Cutoff (Otsu):       {OTSU_MIN_AREA:.2f} px")
print(f"Dynamic Min Intensity Cutoff (99% CumSum): {CUMULATIVE_MIN_INTENSITY:.2f}")
print(f"Static Circularity Limits:          {MIN_CIRCULARITY:.2f} to {MAX_CIRCULARITY:.2f}")

# --- 3. SCORING & FILTERING PIPELINE (STRICT BOUNDARIES) ---
sync_scores = []; sync_status = []
max_observed_area = df['area_px'].max() if not df.empty else 1

for idx, row in df.iterrows():
    area = row['area_px']
    mean_intensity = row['mean_intensity']
    circ_val = row['circularity']

    if area < OTSU_MIN_AREA:
        sync_scores.append(0.0); sync_status.append(f"Rejected (Area {area:.0f}px < Otsu)"); continue
    elif mean_intensity < CUMULATIVE_MIN_INTENSITY:
        sync_scores.append(0.0); sync_status.append(f"Rejected (Intensity {mean_intensity:.1f} < 99% CumSum)"); continue
    elif circ_val < MIN_CIRCULARITY:
        sync_scores.append(0.0); sync_status.append("Rejected (Circularity Too Low)"); continue
    elif circ_val > MAX_CIRCULARITY:
        sync_scores.append(0.0); sync_status.append("Rejected (Circularity Too High)"); continue

    reasons = []
    if area < (OTSU_MIN_AREA * 1.15):
        reasons.append(f"Borderline Area ({area:.0f}px)")
    if mean_intensity < (CUMULATIVE_MIN_INTENSITY * 1.15):
        reasons.append(f"Borderline Intensity ({mean_intensity:.1f})")

    if reasons:
        sync_scores.append(0.0); sync_status.append("Manual Check (" + "; ".join(reasons) + ")"); continue

    area_score = area / max_observed_area

    ideal_int_center = (CUMULATIVE_MIN_INTENSITY + MAX_INTENSITY) / 2
    max_int_dist = max((ideal_int_center - CUMULATIVE_MIN_INTENSITY), 1)
    intensity_score = 1.0 - (abs(mean_intensity - ideal_int_center) / max_int_dist)

    ideal_circ_center = (MIN_CIRCULARITY + MAX_CIRCULARITY) / 2
    max_circ_dist = max((ideal_circ_center - MIN_CIRCULARITY), 1)
    circularity_score = 1.0 - (abs(circ_val - ideal_circ_center) / max_circ_dist)

    final_score = (0.4 * area_score) + (0.35 * intensity_score) + (0.25 * circularity_score)
    sync_scores.append(round(final_score, 3))
    sync_status.append("Accepted Aggregate")

df['Aggregate_Score'] = sync_scores
df['Classification'] = sync_status

# --- 4. CONVERT GRAPHICS BUFFER ---
if 'original_rgb_img' in globals() and original_rgb_img is not None:
    display_img_array = original_rgb_img
else:
    display_img_array = np.zeros((600, 800, 3), dtype=np.uint8)

if display_img_array.dtype != np.uint8:
    img_min, img_max = display_img_array.min(), display_img_array.max()
    norm_img = (((display_img_array - img_min) / max((img_max - img_min), 1)) * 255).astype(np.uint8)
else:
    norm_img = display_img_array

pil_img = Image.fromarray(norm_img)
buff = io.BytesIO()
pil_img.save(buff, format="JPEG")
image_data_uri = f"data:image/jpeg;base64,{base64.b64encode(buff.getvalue()).decode('utf-8')}"

accepted_df = df[df['Classification'] == "Accepted Aggregate"].sort_values(by='Aggregate_Score', ascending=False)
manual_df   = df[df['Classification'].str.contains("Manual Check", na=False)]
rejected_df = df[df['Classification'].str.contains("Rejected", na=False)]

def create_item_list(dataframe, current_bin_id):
    item_list = []
    for idx, row in dataframe.iterrows():
        lbl = int(row['label'])
        item_list.append({
            "roi": str(lbl),
            "area": int(row['area_px']),
            "mean": float(row['mean_intensity']),
            "circ": float(row['circularity']),
            "x": float(row['centroid_x']),
            "y": float(row['centroid_y']),
            "reason": str(row['Classification']).replace("Manual Check (", "").replace("Rejected (", "").replace(")", "") if "(" in str(row['Classification']) else "",
            "starting_bin": current_bin_id,
            "contour": precomputed_contours_json.get(lbl, [])
        })
    return item_list

all_items_json = json.dumps({
    "accepted": create_item_list(accepted_df, 'bin-accepted'),
    "manual": create_item_list(manual_df, 'bin-manual'),
    "rejected": create_item_list(rejected_df, 'bin-rejected')
})

# --- LIVE STATE SYNCHRONIZATION CHANNELS ---
live_lane_assignments = {}
for item in create_item_list(accepted_df, 'bin-accepted'): live_lane_assignments[int(item['roi'])] = 'bin-accepted'
for item in create_item_list(manual_df, 'bin-manual'):    live_lane_assignments[int(item['roi'])] = 'bin-manual'
for item in create_item_list(rejected_df, 'bin-rejected'): live_lane_assignments[int(item['roi'])] = 'bin-rejected'

# Global destination variable accessible everywhere in the notebook
live_accepted_df = pd.DataFrame()
df_display_handle = IPython.display.DisplayHandle()

def update_live_dataframe(label_id, target_lane):
    global live_accepted_df
    live_lane_assignments[int(label_id)] = str(target_lane)

    current_accepted_ids = [lbl for lbl, lane in live_lane_assignments.items() if lane == 'bin-accepted']
    live_accepted_df = df[df['label'].isin(current_accepted_ids)].copy().reset_index(drop=True)

    table_html = f"""
    <div style='margin-top:15px; font-family:Arial; font-size:12px;'>
        <b style='color:#27ae60;'>🟢 Live Kernel DataFrame Synced:</b> {len(live_accepted_df)} items in accepted bin.
        {live_accepted_df.to_html(classes='table', index=False) if not live_accepted_df.empty else '<br><i>(Accepted DataFrame is currently empty)</i>'}
    </div>
    """
    df_display_handle.update(HTML(table_html))

output.register_callback('notebook.UpdateLiveDataFrame', update_live_dataframe)

# --- 5. RENDER DRAGGABLE HUD WEB LAYOUT ---
html_spatial_dashboard = f"""
<style>
    .lane-group {{ flex: 1; display: flex; flex-direction: column; min-width: 220px; }}
    .bin-lane {{ flex-grow: 1; min-height: 250px; max-height: 250px; overflow-y: auto; border-radius: 6px; padding: 10px; display: flex; flex-direction: column; gap: 8px; }}
    .roi-item-card {{ background: #fff; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-size: 12px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); border: 1px solid #ccc; }}
    .roi-item-card:hover {{ border-color: #3498db !important; background: #fdfefe; }}
    .roi-item-card.selected-card {{ background: #e3f2fd !important; border: 2px solid #2196f3 !important; }}
</style>

<div style="font-family: Arial, sans-serif; background: #f1f2f6; padding: 20px; border-radius: 8px; max-width: 1500px; margin: auto; display: flex; flex-direction: column; gap: 20px;">

    <div style="text-align: center;">
        <h3 style="color: #2c3e50; margin: 0 0 5px 0;">🎯 Automated Image Thresholding Dashboard</h3>
        <p style="color: #7f8c8d; margin: 0; font-size:13px;">
            Lanes configured with strict micro-noise cutoff policies. <br>
            <b>Cutoffs:</b> Area Min (Otsu) = {OTSU_MIN_AREA:.1f}px | Intensity Min (99% CumSum) = {CUMULATIVE_MIN_INTENSITY:.1f}
        </p>
    </div>

    <div style="display: flex; flex-direction: row; gap: 20px; align-items: flex-start; flex-wrap: wrap;">

        <div style="flex: 3; min-width: 450px; position: relative; background: #111; padding: 15px; border-radius: 8px; box-shadow: inset 0 0 10px #000; display: flex; justify-content: center; align-items: center;">
            <canvas id="wormCanvas" style="display: block; max-width: 80%; height: auto; border: 1px solid #333; cursor: crosshair;"></canvas>
        </div>

        <div style="flex: 2; min-width: 500px; display: flex; justify-content: space-between; gap: 15px; background: #fff; padding: 15px; border-radius: 8px; border: 1px solid #dcdde1; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">
            <div class="lane-group">
                <h4 style="color: #27ae60; margin: 0 0 8px 0; text-align: center; font-size: 14px;">accepted</h4>
                <div id="bin-accepted" class="bin-lane" style="background: #e8f5e9; border: 2px solid #2ecc71;" ondrop="drop(event)" ondragover="allowDrop(event)"></div>
                <div id="accepted-count" style="text-align: center; font-size: 11px; color: #27ae60; font-weight: bold; margin-top: 5px;">Items: 0</div>
            </div>
            <div class="lane-group">
                <h4 style="color: #f39c12; margin: 0 0 8px 0; text-align: center; font-size: 14px;">manual check</h4>
                <div id="bin-manual" class="bin-lane" style="background: #fff8e1; border: 2px solid #f39c12;" ondrop="drop(event)" ondragover="allowDrop(event)"></div>
                <div id="manual-count" style="text-align: center; font-size: 11px; color: #f39c12; font-weight: bold; margin-top: 5px;">Items: 0</div>
            </div>
            <div class="lane-group">
                <h4 style="color: #c0392b; margin: 0 0 8px 0; text-align: center; font-size: 14px;">rejected</h4>
                <div id="bin-rejected" class="bin-lane" style="background: #ffebee; border: 2px solid #e74c3c;" ondrop="drop(event)" ondragover="allowDrop(event)"></div>
                <div id="rejected-count" style="text-align: center; font-size: 11px; color: #c0392b; font-weight: bold; margin-top: 5px;">Items: 0</div>
            </div>
        </div>

    </div>
</div>

<script>
const masterDataMap = new Map();
const baseImage = new Image();
const canvas = document.getElementById('wormCanvas');
const ctx = canvas.getContext('2d');

const ZOOM_FACTOR = 4;
const ZOOM_BOX_SIZE = 400;
const CROSSHAIR_HIT_RADIUS = 20;

let targetPoint = {{ x: 250, y: 200 }};
let isDraggingCrosshair = false;
let lastHighlightMeta = null;

let zoomBoxOffset = {{ x: -1, y: -1 }};
let isDraggingZoomBox = false;
let zoomDragStartOffset = {{ x: 0, y: 0 }};

baseImage.src = "{image_data_uri}";
baseImage.onload = function() {{
    canvas.width = baseImage.naturalWidth || 800;
    canvas.height = baseImage.naturalHeight || 600;

    if (zoomBoxOffset.x === -1) {{
        zoomBoxOffset.x = canvas.width - ZOOM_BOX_SIZE - 20;
        zoomBoxOffset.y = canvas.height - ZOOM_BOX_SIZE - 20;
    }}
    redrawCanvas();
}};

function getCanvasPoint(event) {{
    const rect = canvas.getBoundingClientRect();
    const scaleX = canvas.width / rect.width;
    const scaleY = canvas.height / rect.height;
    return {{
        x: (event.clientX - rect.left) * scaleX,
        y: (event.clientY - rect.top) * scaleY
    }};
}}

function isOverCrosshairIntersection(cx, cy) {{
    const distance = Math.sqrt((cx - targetPoint.x)**2 + (cy - targetPoint.y)**2);
    return distance <= CROSSHAIR_HIT_RADIUS;
}}

function isOverZoomViewport(cx, cy) {{
    return cx >= zoomBoxOffset.x && cx <= zoomBoxOffset.x + ZOOM_BOX_SIZE &&
           cy >= zoomBoxOffset.y && cy <= zoomBoxOffset.y + ZOOM_BOX_SIZE;
}}

canvas.addEventListener('mousedown', function(e) {{
    const pt = getCanvasPoint(e);

    if (isOverCrosshairIntersection(pt.x, pt.y)) {{
        isDraggingCrosshair = true;
        canvas.style.cursor = 'grabbing';
    }} else if (isOverZoomViewport(pt.x, pt.y)) {{
        isDraggingZoomBox = true;
        zoomDragStartOffset.x = pt.x - zoomBoxOffset.x;
        zoomDragStartOffset.y = pt.y - zoomBoxOffset.y;
        canvas.style.cursor = 'move';
    }}
}});

canvas.addEventListener('mousemove', function(e) {{
    const pt = getCanvasPoint(e);

    if (isDraggingCrosshair) {{
        targetPoint.x = Math.max(0, Math.min(canvas.width, pt.x));
        targetPoint.y = Math.max(0, Math.min(canvas.height, pt.y));
        redrawCanvas();
    }} else if (isDraggingZoomBox) {{
        let newX = pt.x - zoomDragStartOffset.x;
        let newY = pt.y - zoomDragStartOffset.y;

        zoomBoxOffset.x = Math.max(0, Math.min(canvas.width - ZOOM_BOX_SIZE, newX));
        zoomBoxOffset.y = Math.max(0, Math.min(canvas.height - ZOOM_BOX_SIZE, newY));
        redrawCanvas();
    }} else {{
        if (isOverCrosshairIntersection(pt.x, pt.y)) {{
            canvas.style.cursor = 'move';
        }} else if (isOverZoomViewport(pt.x, pt.y)) {{
            canvas.style.cursor = 'grab';
        }} else {{
            canvas.style.cursor = 'crosshair';
        }}
    }}
}});

canvas.addEventListener('mouseup', function(e) {{
    if (isDraggingCrosshair) {{
        isDraggingCrosshair = false;
        canvas.style.cursor = 'move';
        return;
    }}
    if (isDraggingZoomBox) {{
        isDraggingZoomBox = false;
        canvas.style.cursor = 'grab';
        return;
    }}

    const pt = getCanvasPoint(e);
    let matchedId = null;
    let closestDist = 999999;
    masterDataMap.forEach((item, key) => {{
        const d = Math.sqrt((item.x - pt.x) ** 2 + (item.y - pt.y) ** 2);
        const radius = Math.max(5, Math.sqrt(item.area / Math.PI));
        if (d <= radius + 20 && d < closestDist) {{
            closestDist = d;
            matchedId = key;
        }}
    }});
    if (matchedId) highlightDataPoint(matchedId);
}});

canvas.addEventListener('mouseleave', function() {{
    isDraggingCrosshair = false;
    isDraggingZoomBox = false;
}});

function drawAggregateContour(points, strokeStyle, lineWidth) {{
    if (!points || points.length === 0) return;
    ctx.beginPath();
    ctx.moveTo(points[0][0], points[0][1]);
    for (let i = 1; i < points.length; i++) {{
        ctx.lineTo(points[i][0], points[i][1]);
    }}
    ctx.closePath();
    ctx.strokeStyle = strokeStyle;
    ctx.lineWidth = lineWidth;
    ctx.stroke();
}}

function redrawCanvas() {{
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.drawImage(baseImage, 0, 0);

    masterDataMap.forEach((item) => {{
        let contourColor = 'rgba(231, 76, 60, 0.5)';
        if (item.lane === 'bin-accepted')       contourColor = 'rgba(46, 204, 113, 0.8)';
        else if (item.lane === 'bin-manual')   contourColor = 'rgba(243, 156, 18, 0.8)';

        const isSelected = (lastHighlightMeta && lastHighlightMeta.roi === item.roi);
        const finalLineWidth = isSelected ? 1.5 : 1.5;

        drawAggregateContour(item.contour, contourColor, finalLineWidth);
    }});

    drawTrackingCrosshair();
    drawCornerZoomViewport();
}}

function drawTrackingCrosshair() {{
    ctx.save();
    ctx.strokeStyle = '#00d2ff';
    ctx.lineWidth = 1;

    ctx.beginPath();
    ctx.moveTo(0, targetPoint.y);
    ctx.lineTo(canvas.width, targetPoint.y);
    ctx.stroke();

    ctx.beginPath();
    ctx.moveTo(targetPoint.x, 0);
    ctx.lineTo(targetPoint.x, canvas.height);
    ctx.stroke();
    ctx.restore();
}}

function drawCornerZoomViewport() {{
    ctx.save();

    const bx = zoomBoxOffset.x;
    const by = zoomBoxOffset.y;

    ctx.strokeStyle = '#00d2ff';
    ctx.lineWidth = 1.5;
    ctx.setLineDash([4, 4]);
    ctx.beginPath();
    ctx.moveTo(targetPoint.x, targetPoint.y);
    ctx.lineTo(bx + ZOOM_BOX_SIZE / 2, by + ZOOM_BOX_SIZE / 2);
    ctx.stroke();
    ctx.setLineDash([]);

    const srcSize = ZOOM_BOX_SIZE / ZOOM_FACTOR;
    const srcX = targetPoint.x - srcSize / 2;
    const srcY = targetPoint.y - srcSize / 2;

    ctx.fillStyle = '#111';
    ctx.fillRect(bx, by, ZOOM_BOX_SIZE, ZOOM_BOX_SIZE);

    ctx.imageSmoothingEnabled = false;
    ctx.drawImage(canvas, srcX, srcY, srcSize, srcSize, bx, by, ZOOM_BOX_SIZE, ZOOM_BOX_SIZE);

    ctx.strokeStyle = 'rgba(0, 210, 255, 0.85)';
    ctx.lineWidth = 1;
    const zoomCenterY = by + (ZOOM_BOX_SIZE / 2);
    const zoomCenterX = bx + (ZOOM_BOX_SIZE / 2);

    ctx.beginPath();
    ctx.moveTo(bx, zoomCenterY);
    ctx.lineTo(bx + ZOOM_BOX_SIZE, zoomCenterY);
    ctx.stroke();

    ctx.beginPath();
    ctx.moveTo(zoomCenterX, by);
    ctx.lineTo(zoomCenterX, by + ZOOM_BOX_SIZE);
    ctx.stroke();

    ctx.strokeStyle = '#00d2ff';
    ctx.lineWidth = 3;
    ctx.strokeRect(bx, by, ZOOM_BOX_SIZE, ZOOM_BOX_SIZE);

    ctx.fillStyle = '#00d2ff';
    ctx.font = 'bold 11px Arial';
    ctx.fillText('🎛️ DRAGGABLE ZOOM VIEWPORT (' + ZOOM_FACTOR + 'x)', bx + 8, by + 18);
    ctx.restore();
}}

function highlightDataPoint(cardId) {{
    document.querySelectorAll('.roi-item-card').forEach(c => c.classList.remove('selected-card'));
    const targetCard = document.getElementById(cardId);
    if (!targetCard) return;
    targetCard.classList.add('selected-card');
    targetCard.scrollIntoView({{ behavior: 'smooth', block: 'nearest' }});

    const meta = masterDataMap.get(cardId);
    if (meta) {{
        lastHighlightMeta = meta;
        targetPoint.x = meta.x;
        targetPoint.y = meta.y;
        redrawCanvas();
    }}
}}

function allowDrop(ev) {{ ev.preventDefault(); }}

function drop(ev) {{
    ev.preventDefault();
    const id = ev.dataTransfer.getData('text/plain');
    const draggedElement = document.getElementById(id);
    let targetLane = ev.target;
    while (targetLane && !targetLane.classList.contains('bin-lane')) {{
        targetLane = targetLane.parentElement;
    }}
    if (targetLane && draggedElement) {{
        targetLane.appendChild(draggedElement);
        updateCounters();
        const meta = masterDataMap.get(id);
        if (meta) {{
            meta.lane = targetLane.id;
            redrawCanvas();

            const rawLabel = id.split('-')[1];
            google.colab.kernel.invokeFunction('notebook.UpdateLiveDataFrame', [rawLabel, targetLane.id], {{}});
        }}
    }}
}}

function updateCounters() {{
    document.getElementById('accepted-count').innerText = 'Items: ' + document.getElementById('bin-accepted').querySelectorAll('.roi-item-card').length;
    document.getElementById('manual-count').innerText   = 'Items: ' + document.getElementById('bin-manual').querySelectorAll('.roi-item-card').length;
    document.getElementById('rejected-count').innerText = 'Items: ' + document.getElementById('bin-rejected').querySelectorAll('.roi-item-card').length;
}}

(function() {{
    const dataPayload = {all_items_json};

    function parseAndPopulate(items, laneId, prefix, baseColor) {{
        const laneElement = document.getElementById(laneId);
        items.forEach((item) => {{
            const uniqueId = `${{prefix}}-${{item.roi}}`;
            masterDataMap.set(uniqueId, {{
                roi: item.roi,
                x: item.x,
                y: item.y,
                area: item.area,
                lane: laneId,
                contour: item.contour
            }});

            const card = document.createElement('div');
            card.id = uniqueId;
            card.className = 'roi-item-card';
            card.draggable = true;
            card.style.borderLeft = `5px solid ${{baseColor}}`;

            card.innerHTML = `
                <strong style="color: #000000;">Label: ${{item.roi}}</strong>
                <span style="font-size:10px; color:#7f8c8d;">${{item.reason ? '(' + item.reason + ')' : ''}}</span><br>
                <span style="font-size:11px; color:#555;">
                    📐 Area: <b>${{item.area}}</b> &nbsp;|&nbsp;
                    💡 Mean: <b>${{item.mean.toFixed(1)}}</b> &nbsp;|&nbsp;
                    ⭕ Circ: <b>${{item.circ.toFixed(2)}}</b>
                </span>`;

            card.addEventListener('click', () => highlightDataPoint(uniqueId));
            card.addEventListener('dragstart', (ev) => {{ ev.dataTransfer.setData('text/plain', ev.target.id); }});
            laneElement.appendChild(card);
        }});
    }}

    parseAndPopulate(dataPayload.accepted, 'bin-accepted', 'acc', '#2ecc71');
    parseAndPopulate(dataPayload.manual,   'bin-manual',   'man', '#f39c12');
    parseAndPopulate(dataPayload.rejected, 'bin-rejected', 'rej', '#e74c3c');
    updateCounters();
}})();
</script>
"""

# Render dashboard UI and link its live reflection table right underneath
display(HTML(html_spatial_dashboard))
df_display_handle.display(HTML("<div style='font-family:Arial; font-size:12px; color:#7f8c8d;'><i>Initializing real-time notebook data-sync link...</i></div>"))

# Populate the baseline data frame immediately upon rendering
for lbl, lane in live_lane_assignments.items():
    update_live_dataframe(lbl, lane)

In [ ]:
#@title Export data to a csv file

# Assume df is available from the previous cells (EL8NPacFmukc).
# Ensure 'df' DataFrame exists and contains 'label', 'area_px', 'mean_intensity' columns.

# Create a DataFrame for export with the desired column order and data.
# Column 1: 'label' data from df, with no header.
# Column 2: 'area_px' data from df, with 'Area' as header.
# Column 3: 'mean_intensity' data from df, with 'Mean' as header.
export_df = pd.DataFrame({
    '': live_accepted_df['label'].astype(int),        # Column 1: 'label' data, with an empty header
    'Area': live_accepted_df['area_px'],               # Column 2: 'area_px' data
    'Mean': live_accepted_df['mean_intensity']         # Column 3: 'mean_intensity' data
})

output_filename = 'aggregate_data.csv'

# Export to CSV.
# index=False is still used as the DataFrame index is not relevant for the CSV output.
# header=True (default behavior) will use the column names from export_df.
export_df.to_csv(output_filename, index=False)

print(f"Data exported to '{output_filename}' successfully.")